**`04_detect_phantom_parcel_ids`**

Identify non-taxable placeholder values in `parcel_id_assessor` (e.g., `WATER`,
`ROW`, `PRIV_ROW`) that should be added to the `exclude_by_value` step in
`US_parcel-openplaces-2026.yaml`.

The `exclude_by_value` step filters out placeholder rows that do not represent
real taxable parcels (e.g., `WATER` or road rights-of-way like `ROW` in Carteret
County, NC; `PRIV_ROW` in Massachusetts counties). In historical surveys, these
rows consistently have empty `use_group_combined` and zero or missing land and
improvement values.

Filtering by keywords in `use_group_combined` is **not** safe. Real, valuable
parcels (such as waterfront lots or condominiums) often contain water- or
road-related descriptions in their use-code text. A simple substring search
would delete thousands of legitimate parcels.

Instead, the most reliable indicator of a placeholder is the `parcel_id_assessor`
value itself. Since these names vary across data sources, they must be audited
and added to the recipe manually.

Use this notebook to audit curated parcel outputs from a newly ingested county.
Review the flagged candidates, and manually add verified placeholder IDs to the
recipe's `exclude_by_value` list. This notebook is read-only and does not write
to disk.

In [ ]:
import argparse

import pandas as pd

import openplaces as op

In [ ]:
# Define CLI arguments for the audit
parser = argparse.ArgumentParser(
    description='Detect candidate placeholder assessor parcel IDs'
)

parser.add_argument(
    '--admin_id',
    default='US-NC-CE',
    help='Admin unit whose curated parcel output to audit',
)
parser.add_argument(
    '--recipe_id',
    default='US_parcel-openplaces-2026',
    help='Curated parcel recipe ID',
)
parser.add_argument(
    '--min_count',
    type=int,
    default=10,
    help='Minimum recurrence count for a parcel_id_assessor value to be flagged',
)

# Set up test arguments (simulating CLI parameters)

In [ ]:
ARGS_TEST = '--admin_id US-NC-CE --recipe_id US_parcel-openplaces-2026 --min_count 10'

# Split argument string to simulate command-line inputs
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse the simulated arguments
args = parser.parse_args(args_list)

# Verify the parsed arguments
args

# Load curated parcel data

In [ ]:
columns = [
    'parcel_id_assessor',
    'use_group_combined',
    'improvement_value',
    'land_value',
]
parcels = op.get_entities(args.recipe_id, admin_id=args.admin_id, columns=columns)
len(parcels)

# Detect placeholder candidates

A `parcel_id_assessor` value is considered a candidate placeholder if it:
1. Repeats more than `--min_count` times (legitimate assessor IDs are typically unique, while placeholders are reused).
2. Contains non-numeric characters.
3. Co-occurs (in all instances) with an empty `use_group_combined` and zero/missing `improvement_value` and `land_value`.

This signature matches confirmed placeholders (like `WATER` and `ROW`) while sparing legitimate large-area parcels (such as Cape Lookout National Seashore, `CALO`), which hold actual valuation and populated use codes.

In [ ]:
def detect_phantom_parcel_ids(
    parcels: pd.DataFrame, min_count: int = 10
) -> pd.DataFrame:
    # Get counts of all unique assessor IDs
    ids = parcels['parcel_id_assessor'].astype('string')
    counts = ids.value_counts()

    # Filter for non-numeric IDs that repeat more than the minimum threshold
    recurring = counts[counts > min_count]
    non_numeric = recurring[~recurring.index.str.fullmatch(r'\d+', na=False)]

    # Identify rows with empty use codes and zero values
    use_blank = parcels['use_group_combined'].isna() | parcels[
        'use_group_combined'
    ].astype('string').str.strip().eq('')
    value_zero = parcels['improvement_value'].fillna(0).eq(0) & parcels[
        'land_value'
    ].fillna(0).eq(0)
    phantom_signature = use_blank & value_zero

    # Keep candidates where every occurrence matches the phantom signature
    rows = []
    for value, count in non_numeric.items():
        subset = ids.eq(value)
        if phantom_signature[subset].all():
            rows.append({'parcel_id_assessor': value, 'count': count})

    columns = pd.Index(['parcel_id_assessor', 'count'])
    return (
        pd.DataFrame(rows, columns=columns)
        .sort_values('count', ascending=False)
        .reset_index(drop=True)
    )


candidates = detect_phantom_parcel_ids(parcels, min_count=args.min_count)
candidates

# Next steps

Manually review the identified `candidates`. This script flags potential placeholders, but some could be legitimate (e.g., a source might reuse a single non-numeric ID for a group of real, zero-value parcels).

For any confirmed placeholders you wish to filter out:
1. Open the recipe file `US_parcel-openplaces-2026.yaml`.
2. Add the value to the `exclude_by_value` list.
3. Add a brief comment documenting the county or source where the placeholder was identified.